In [30]:
from collections import Counter, defaultdict
import matplotlib.pyplot as plt
import numpy as np
import spacy
from spacy.matcher import DependencyMatcher
from spacy.displacy import parse_deps
import spacy.displacy as displacy
import sys
import tqdm
import random

sys.path.append("../src")

from shared import tqdm_readlines

In [2]:
deu_nlp = spacy.load("de_zdl_lg")

In [3]:
deu_snts = dict([(ln.strip().split("\t")[0], ln.strip().split("\t")[1]) for ln in tqdm_readlines("../data/leipzig/deu.tsv")])

100%|█████████████████████████████████████████████████████| 100000/100000 [00:00<00:00, 1374947.22it/s]


In [38]:
from spacy.tokens import DocBin

# doc_bin = DocBin(attrs=["LEMMA", "ENT_IOB", "ENT_TYPE"], store_user_data=True)
doc_bin = DocBin(store_user_data=True)

for doc in deu_nlp.pipe(tqdm.tqdm(list(deu_snts.values()))):
    doc_bin.add(doc)

print("DONE")

bytes_data = doc_bin.to_bytes()





  0%|                                                                       | 0/100000 [00:00<?, ?it/s]



  1%|▌                                                           | 1000/100000 [00:23<38:40, 42.66it/s]



  1%|▌                                                           | 1000/100000 [00:36<38:40, 42.66it/s]



  2%|█▏                                                          | 2000/100000 [00:47<38:52, 42.02it/s]



  2%|█▍                                                          | 2332/100000 [00:47<30:23, 53.56it/s]



  3%|█▋                                                          | 2845/100000 [00:47<20:30, 78.95it/s]



  3%|█▋                                                          | 2845/100000 [01:06<20:30, 78.95it/s]



  3%|█▊                                                          | 3000/100000 [01:13<51:20, 31.49it/s]



  4%|██▏                                                         | 3542/100000 [01:13<31:44, 50.65it/s]



  4%|██▏                         

DONE


In [39]:
f = open("../dump/leipzig_deu_bytes", "wb")
f.write(bytes_data)
f.close()

In [4]:
from spacy.tokens import DocBin

bytes_data = open("../dump/leipzig_deu_bytes", "rb").read()
doc_bin = DocBin().from_bytes(bytes_data)
docs = list(doc_bin.get_docs(deu_nlp.vocab))

In [40]:
def search_nlp(nlp, ptn):
    matcher = DependencyMatcher(nlp.vocab)
    matcher.add("PTN", [ptn])
    return matcher

zu_match = search_nlp(deu_nlp, [{
    "RIGHT_ID": "anchor", 
    "RIGHT_ATTRS": { "POS": "AUX" }
}, {
    "LEFT_ID": "anchor", 
    "REL_OP": ">", 
    "RIGHT_ID": "comp_verb", 
    "RIGHT_ATTRS": {
        "POS": "VERB",
        "DEP": "xcomp"
    }
}, {
    "LEFT_ID": "comp_verb", 
    "REL_OP": ">", 
    "RIGHT_ID": "zu", 
    "RIGHT_ATTRS": {
        "POS": "PART",
        "DEP": "mark",
        "LEMMA": "zu"
    }
}])

haben_zu_match = search_nlp(deu_nlp, [{
    "RIGHT_ID": "anchor", 
    "RIGHT_ATTRS": { 
        "POS": "AUX",
        "LEMMA": "haben"
    }
}, {
    "LEFT_ID": "anchor", 
    "REL_OP": ">", 
    "RIGHT_ID": "haben_obj", 
    "RIGHT_ATTRS": {
        "POS": "NOUN",
        "DEP": "obj"
    }
}, {
    "LEFT_ID": "haben_obj", 
    "REL_OP": ">", 
    "RIGHT_ID": "comp_verb", 
    "RIGHT_ATTRS": {
        "POS": "VERB",
        "DEP": "xcomp"
    }
}, {
    "LEFT_ID": "comp_verb", 
    "REL_OP": ">", 
    "RIGHT_ID": "zu", 
    "RIGHT_ATTRS": {
        "POS": "PART",
        "DEP": "mark",
        "LEMMA": "zu"
    }
}])

auxp_match = search_nlp(deu_nlp, [{
    "RIGHT_ID": "anchor", 
    "RIGHT_ATTRS": { "POS": "VERB" }
}, {
    "LEFT_ID": "anchor", 
    "REL_OP": ">", 
    "RIGHT_ID": "obl_obj", 
    "RIGHT_ATTRS": {
        "POS": "NOUN",
        "DEP": { "IN": ["obl", "obj"]}
    }
}, {
    "LEFT_ID": "obl_obj", 
    "REL_OP": ">", 
    "RIGHT_ID": "obl_prep", 
    "RIGHT_ATTRS": {
        "POS": "ADP",
        "DEP": "case"
    }
}])

# s = "Das wird von den Fakten abhängen."#
s = "Von Ihrer Antwort wird abhängen, wie wir morgen abstimmen."
# s = "Das sollte vom Dossier abhängen."
# s = "Ich habe kein Interesse daran, das zu tun."
# s = "Ich habe keinen Grund, das zu tun."
# s = "Ich habe nicht vor, das zu tun."
print(parse_deps(deu_nlp(s)))
auxp_match(deu_nlp(s))

{'words': [{'text': 'Von', 'tag': 'ADP', 'lemma': None}, {'text': 'Ihrer', 'tag': 'DET', 'lemma': None}, {'text': 'Antwort', 'tag': 'NOUN', 'lemma': None}, {'text': 'wird', 'tag': 'AUX', 'lemma': None}, {'text': 'abhängen,', 'tag': 'VERB', 'lemma': None}, {'text': 'wie', 'tag': 'ADV', 'lemma': None}, {'text': 'wir', 'tag': 'PRON', 'lemma': None}, {'text': 'morgen', 'tag': 'ADV', 'lemma': None}, {'text': 'abstimmen.', 'tag': 'VERB', 'lemma': None}], 'arcs': [{'start': 0, 'end': 2, 'label': 'case', 'dir': 'left'}, {'start': 1, 'end': 2, 'label': 'det', 'dir': 'left'}, {'start': 2, 'end': 4, 'label': 'obj', 'dir': 'left'}, {'start': 3, 'end': 4, 'label': 'aux', 'dir': 'left'}, {'start': 5, 'end': 8, 'label': 'advmod', 'dir': 'left'}, {'start': 6, 'end': 8, 'label': 'nsubj', 'dir': 'left'}, {'start': 7, 'end': 8, 'label': 'advmod', 'dir': 'left'}, {'start': 4, 'end': 8, 'label': 'acl', 'dir': 'right'}], 'settings': {'lang': 'de', 'direction': 'ltr'}}


[(8884005714813011517, [4, 2, 0])]

In [34]:
zu_c = Counter()
haben_zu_c = Counter()
haben_zu_dict = defaultdict(list)

for doc in docs:
    ms = zu_match(doc)
    for m in ms:
        # print(doc)
        zu_c[doc[m[1][0]].lemma_] += 1

    ms = haben_zu_match(doc)
    for m in ms:
        # print(doc)
        lem = doc[m[1][1]].lemma_
        haben_zu_c[lem] += 1
        haben_zu_dict[lem] = haben_zu_dict[lem] + [doc]

for s, n in haben_zu_c.most_common():
    print(n, "\t", s)
    for doc in haben_zu_dict[s][:3]:
        print("\t", doc)
    

23 	 Möglichkeit
	 Altkleiderverwerter hätten dann die Möglichkeit, Kleidung besser und schneller zu sortieren.
	 Außerdem haben Unternehmen die Möglichkeit, Boot Camps in ihren Räumen durchführen zu lassen.
	 Berufstätige Personen haben durch ein berufsbegleitendes Studium die Möglichkeit, ihre Kenntnisse und Fähigkeiten in ihrem Fachbereich zu vertiefen und ihre Karriere voranzutreiben.
18 	 Chance
	 Auch wir hatten Chancen, das Spiel zu gewinnen.
	 «Damit habe ich viel bessere Chancen, eine Stelle in Namibia zu bekommen.»
	 Der Erlebnistag richtet sich vornehmlich an den Tischtennis-Nachwuchs, der am Sonntag die Chance hat, mit dem 54-jährigen Welt- und Europameister gemeinsam zu trainieren.
4 	 Anspruch
	 Aber wir müssen natürlich den Anspruch haben, mindestens European League zu spielen, sonst bekommst du die Spieler nicht, die du bekommen willst.
	 Der Sozialarbeiter hatte dabei nie den Anspruch, die Fantasy neu zu erfinden.
	 Man habe den Anspruch, den Landeshauptmann zu stellen